# Generators in Python — pause, remember, continue

> **Explain it like I am five:** Imagine a chef who makes one pancake only when you ask. The chef does not fill the whole table first. A generator makes one value at a time in the same way.

A generator is a convenient way to create an iterator. It uses `yield` to produce a value, pause, and remember where it stopped.

## Learning goals

- understand `yield` and lazy evaluation;
- compare `yield` with `return`;
- consume generators with `next()` and `for`;
- write generator expressions and pipelines;
- process large or infinite sequences safely.


## 1. The original square generator

Calling a generator function does **not** run its body immediately. It creates a generator object. Work begins when `next()` or a loop asks for a value.

The original example is kept exactly below.


In [1]:
def square(n):
    for i in range(3):
        yield i**2


In [2]:
square(3)


<generator object square at 0x00000206A4F42A80>

The result looks like `<generator object ...>` because no values have been requested yet.


In [3]:
for i in square(3):
    print(i)


0
1
4


In [4]:
a=square(3)
a


<generator object square at 0x00000206A4F428E0>

In [5]:
next(a)


0

Each `next(a)` resumes the function until the next `yield`. Then it pauses again.

### A useful bug to notice

The function accepts `n`, but the original loop says `range(3)`. Therefore `square(100)` still gives only three values. Here is the general version:


In [6]:
def squares(n):
    """Yield the squares of 0 through n - 1."""
    for number in range(n):
        yield number ** 2


print(list(squares(5)))


[0, 1, 4, 9, 16]


## 2. A generator pauses at every `yield`

This is the original three-value example. It is like a story with three pause buttons.


In [7]:
def my_generator():
    yield 1
    yield 2
    yield 3


In [8]:
gen=my_generator()
gen


<generator object my_generator at 0x00000206A4E19C00>

In [9]:
next(gen)


1

In [10]:
for val in gen:
    print(val)


2
3


Notice that the loop printed `2` and `3`, not `1`. The earlier `next(gen)` already consumed `1`. After the loop, `gen` is exhausted.


In [11]:
print(list(gen))  # Empty because gen is finished.

fresh_gen = my_generator()
print(list(fresh_gen))


[]
[1, 2, 3]


## 3. `yield` vs. `return`

| `return` | `yield` |
|---|---|
| Sends one final result | Sends one value and pauses |
| Ends the function | Remembers local variables and position |
| Normal function call returns the result | Generator function call returns a generator object |

A generator may use `return` with no value to stop early.


In [12]:
def countdown(start):
    while start > 0:
        yield start
        start -= 1
    return  # Ends the generator; iteration sees StopIteration.


print(list(countdown(5)))


[5, 4, 3, 2, 1]


## 4. See the pause-and-resume behavior

The print statements show exactly when generator code runs.


In [13]:
def tiny_story():
    print("A: starting")
    yield "first gift"
    print("B: resumed")
    yield "second gift"
    print("C: finished")


story = tiny_story()
print("Generator created; body has not started.")
print(next(story))
print(next(story))

try:
    next(story)
except StopIteration:
    print("No more gifts.")


Generator created; body has not started.
A: starting
first gift
B: resumed
second gift
C: finished
No more gifts.


## 5. Generator expressions

A generator expression looks like a list comprehension with round brackets.

- List comprehension: `[x * x for x in values]` — builds every result now.
- Generator expression: `(x * x for x in values)` — produces results later, one by one.


In [14]:
square_list = [number ** 2 for number in range(5)]
square_generator = (number ** 2 for number in range(5))

print(square_list)
print(square_generator)
print(list(square_generator))


[0, 1, 4, 9, 16]
<generator object <genexpr> at 0x00000206A4FD1560>
[0, 1, 4, 9, 16]


## 6. Memory: store everything or make values on demand

`sys.getsizeof` does not measure every nested object, but it gives a simple comparison: a million-item list is much larger than a generator recipe.


In [15]:
import sys

large_list = [number ** 2 for number in range(100_000)]
large_generator = (number ** 2 for number in range(100_000))

print(f"List object: {sys.getsizeof(large_list):,} bytes")
print(f"Generator object: {sys.getsizeof(large_generator):,} bytes")


List object: 800,984 bytes
Generator object: 208 bytes


## 7. Practical example: reading a large file

A file object already supports line-by-line iteration. Wrapping it in a generator lets us add processing while still keeping only one line at a time in memory.

**Original function:**


In [16]:
### Practical : Reading LArge Files

def read_large_file(file_path):
    with open(file_path,'r') as file:
        for line in file:
            yield line


The original course uses `large_file.txt`. The next cell creates a tiny temporary demonstration file with that name, runs the original loop, and cleans it up afterward so the notebook works on a fresh machine.


In [17]:
from pathlib import Path

file_path='large_file.txt'
demo_file = Path(file_path)
demo_file.write_text("first line\nsecond line\nthird line\n", encoding="utf-8")

try:
    for line in read_large_file(file_path):
        print(line.strip())
finally:
    demo_file.unlink(missing_ok=True)


first line
second line
third line


## 8. Generator pipelines

Generators can form an assembly line. Each stage handles one item and passes it to the next stage. This is excellent for large data because the whole result does not need to exist at once.


In [18]:
raw_values = [" 10 ", "skip", " 25 ", "-3", " 8 "]

stripped = (value.strip() for value in raw_values)
numeric_text = (value for value in stripped if value.lstrip("-").isdigit())
numbers = (int(value) for value in numeric_text)
positive_squares = (number ** 2 for number in numbers if number > 0)

print(list(positive_squares))


[100, 625, 64]


## 9. `yield from` delegates to another iterable

`yield from values` means “yield every item from `values`.” It is cleaner than writing a small forwarding loop.


In [19]:
def chain_groups(*groups):
    for group in groups:
        yield from group


print(list(chain_groups([1, 2], (3, 4), "Hi")))


[1, 2, 3, 4, 'H', 'i']


## 10. Infinite generators

An infinite generator is safe only when the consumer knows when to stop. Here, `islice` takes five Fibonacci numbers.


In [20]:
from itertools import islice

def fibonacci():
    first, second = 0, 1
    while True:
        yield first
        first, second = second, first + second


print(list(islice(fibonacci(), 10)))


[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


## 11. Common mistakes

- **Reusing an exhausted generator:** call the generator function again to make a fresh one.
- **Forgetting that code is lazy:** errors inside the body may appear only when iteration starts.
- **Calling `len(generator)`:** generators usually have no known length. Consume them only if appropriate, or count as you iterate.
- **Using a generator when repeated access is required:** use a list if the data is small and must be indexed or reused many times.
- **Accidentally consuming it while debugging:** `list(gen)` uses all remaining values.
- **Leaving an infinite generator unbounded:** combine it with a stop condition or `itertools.islice`.


## 12. Mini practice

Write the answer first, then compare it with the solution cell.

1. Yield even numbers from `0` up to (but not including) `limit`.
2. Flatten several small lists using `yield from`.
3. Explain why the second `list(cubes)` below is empty.


In [21]:
def even_numbers(limit):
    for number in range(limit):
        if number % 2 == 0:
            yield number


print(list(even_numbers(10)))

cubes = (number ** 3 for number in range(4))
print(list(cubes))
print(list(cubes))  # Same generator, already exhausted.


[0, 2, 4, 6, 8]
[0, 1, 8, 27]
[]


## Easy revision cheat sheet

| Need | Pattern | Meaning |
|---|---|---|
| Create generator function | `def g(): yield value` | Pause after each value |
| Create generator expression | `(f(x) for x in items)` | Lazy comprehension |
| Get one value | `next(gen)` | Resume until next `yield` |
| Consume safely | `for value in gen:` | Stops automatically |
| Delegate | `yield from iterable` | Forward every item |
| Restart | `gen = g()` | Generators do not rewind |
| Bound infinity | `islice(gen, n)` | Take only `n` items |

**One-sentence memory trick:** A normal function gives one final box with `return`; a generator hands you one item, pauses at `yield`, and continues later.

## Conclusion

Iterators define the one-item-at-a-time protocol. Generators are the easiest way to create iterators because Python manages the saved position and `StopIteration` for us. Use them for large files, data pipelines, streamed results, or sequences that would be wasteful—or impossible—to store all at once.
